# Argos — Kronos Forecasting Host

This notebook loads Kronos (candlestick foundation model) and exposes it two ways:

1. **Batch precompute** — run once over your full backtest date range, dump forecasts + uncertainty to a JSON file in Drive. Best for walk-forward backtesting since dates are known ahead of time.
2. **Live API** — a small FastAPI app tunneled with ngrok, so your multi-agent orchestrator (running elsewhere) can hit Kronos on demand during active dev.

Runtime: make sure **Runtime → Change runtime type → T4 GPU** is selected before running.

## 1. Setup — clone repo, install deps

In [ ]:
!git clone https://github.com/shiyu-coder/Kronos.git
%cd Kronos
!pip install -q -r requirements.txt
!pip install -q fastapi uvicorn pyngrok nest_asyncio pandas numpy

## 2. Load model + tokenizer from Hugging Face Hub

In [ ]:
import torch
from model import Kronos, KronosTokenizer, KronosPredictor

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

# Swap for Kronos-small / Kronos-base / Kronos-large per your speed/quality tradeoff
MODEL_NAME = "NeoQuasar/Kronos-base"
TOKENIZER_NAME = "NeoQuasar/Kronos-Tokenizer-base"

tokenizer = KronosTokenizer.from_pretrained(TOKENIZER_NAME)
model = Kronos.from_pretrained(MODEL_NAME)

predictor = KronosPredictor(model, tokenizer, device=DEVICE)
print("Kronos loaded.")

## 3. Forecast-with-uncertainty helper

Runs multiple sampled forecasts for the same lookback window and returns the point forecast plus a dispersion metric your Risk Manager agent can use for veto/sizing logic.

In [ ]:
import numpy as np
import pandas as pd

def forecast_with_uncertainty(ohlcv_df: pd.DataFrame, pred_len: int = 10, n_samples: int = 15,
                                T: float = 1.0, top_p: float = 0.9):
    """
    ohlcv_df: DataFrame with columns ['open','high','low','close','volume'] and a
              DatetimeIndex (or a 'timestamps' column — see Kronos README for exact schema).
              Max lookback length is 512 rows; longer inputs are auto-truncated.
    pred_len: number of future candles to forecast.
    n_samples: number of sampled forecast paths to draw for the uncertainty estimate.

    Returns a dict with the mean forecast path and a per-step std-dev (uncertainty) series.
    """
    paths = []
    for _ in range(n_samples):
        pred_df = predictor.predict(
            df=ohlcv_df,
            pred_len=pred_len,
            T=T,
            top_p=top_p,
            sample_count=1,
        )
        paths.append(pred_df["close"].values)

    paths = np.stack(paths)  # shape: (n_samples, pred_len)
    mean_path = paths.mean(axis=0)
    std_path = paths.std(axis=0)

    return {
        "mean_forecast": mean_path.tolist(),
        "uncertainty": std_path.tolist(),
        "uncertainty_normalized": (std_path / (np.abs(mean_path) + 1e-8)).tolist(),
        "n_samples": n_samples,
        "pred_len": pred_len,
    }

## 4. Batch precompute (recommended for walk-forward backtesting)

Loop over your fixed backtest dates and tickers, save everything to one JSON in Drive. Your orchestrator then just does a dict lookup — no live dependency during the actual backtest run.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json

# --- EDIT THESE ---
TICKERS = ["AAPL"]              # your 1-3 fixed tickers
BACKTEST_DATES = []             # list of pd.Timestamp / str dates you'll evaluate on
LOOKBACK = 512                  # max supported by Kronos-small/base
OUTPUT_PATH = "/content/drive/MyDrive/argos_kronos_forecasts.json"
# You'll need your own OHLCV loader here — replace load_ohlcv with however
# you're pulling price history elsewhere in the project.

def load_ohlcv(ticker: str, as_of_date, lookback: int) -> pd.DataFrame:
    raise NotImplementedError("Wire this up to your existing OHLCV data source")

results = {}
for ticker in TICKERS:
    results[ticker] = {}
    for date in BACKTEST_DATES:
        df = load_ohlcv(ticker, date, LOOKBACK)
        results[ticker][str(date)] = forecast_with_uncertainty(df)

with open(OUTPUT_PATH, "w") as f:
    json.dump(results, f, indent=2)

print(f"Saved {sum(len(v) for v in results.values())} forecasts to {OUTPUT_PATH}")

## 5. Live API (FastAPI + ngrok tunnel)

For on-demand calls from your multi-agent orchestrator during active dev. Requires a free ngrok authtoken — get one at https://dashboard.ngrok.com/get-started/your-authtoken and paste it below.

**Note:** this tunnel dies when the Colab session ends/disconnects. Treat it as a dev-time convenience, not a production dependency.

In [ ]:
NGROK_AUTHTOKEN = ""  # <-- paste your token here

from pyngrok import ngrok, conf
conf.get_default().auth_token = NGROK_AUTHTOKEN

In [ ]:
import nest_asyncio
nest_asyncio.apply()

from fastapi import FastAPI
from pydantic import BaseModel
from typing import List
import uvicorn

app = FastAPI(title="Argos Kronos Forecast Service")

class Candle(BaseModel):
    timestamp: str
    open: float
    high: float
    low: float
    close: float
    volume: float

class ForecastRequest(BaseModel):
    ticker: str
    candles: List[Candle]   # up to 512, most recent last
    pred_len: int = 10
    n_samples: int = 15

@app.get("/health")
def health():
    return {"status": "ok", "device": DEVICE}

@app.post("/forecast")
def forecast(req: ForecastRequest):
    df = pd.DataFrame([c.dict() for c in req.candles])
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.set_index("timestamp")
    result = forecast_with_uncertainty(df, pred_len=req.pred_len, n_samples=req.n_samples)
    result["ticker"] = req.ticker
    return result

In [ ]:
public_url = ngrok.connect(8000)
print("Kronos live endpoint:", public_url)
print("POST to:", str(public_url) + "/forecast")
print("Keep this cell's runtime alive — closing the notebook or losing the Colab session kills the tunnel.")

uvicorn.run(app, host="0.0.0.0", port=8000)

## 6. Quick local test (run before exposing via ngrok)

Sanity-check the forecast function on dummy data before wiring up the live endpoint or batch loop.

In [ ]:
import numpy as np

dummy_idx = pd.date_range(end=pd.Timestamp.today(), periods=200, freq="D")
dummy = pd.DataFrame({
    "open": np.random.uniform(95, 105, 200),
    "high": np.random.uniform(100, 110, 200),
    "low": np.random.uniform(90, 100, 200),
    "close": np.random.uniform(95, 105, 200),
    "volume": np.random.uniform(1e6, 5e6, 200),
}, index=dummy_idx)

out = forecast_with_uncertainty(dummy, pred_len=5, n_samples=5)
print(json.dumps(out, indent=2))